In [1]:
%pip install numpy pandas streamlit matplotlib scikit-learn


  Using cached scipy-1.17.1-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 57.3 MB/s  0:00:00
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.17.1-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (35.2 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


Percentiles from LMS are computed in two steps: first convert the observed value \(X\) to a z-score using the age-specific \(L, M, S\) values, then convert that z-score to a percentile using the standard normal distribution. [cdc](https://www.cdc.gov/growthcharts/cdc-data-files.htm)

## Formula

If \(L \neq 0\):

\[
z = \frac{(X/M)^L - 1}{L \cdot S}
\]

If \(L = 0\):

\[
z = \frac{\ln(X/M)}{S}
\]

Then:

\[
\text{percentile} = 100 \times \Phi(z)
\]

where \(\Phi(z)\) is the cumulative distribution function of the standard normal distribution. [peteredewitt](https://www.peteredewitt.com/pedbp/articles/growth-standards.html)

## Intuition

- \(M\) is the median for that age.
- \(S\) controls spread.
- \(L\) adjusts for skewness via a Box-Cox-style transformation. [cdc](https://www.cdc.gov/growthcharts/cdc-data-files.htm)

So if the observed VO2 is exactly equal to \(M\), then \(z = 0\) and the percentile is 50th. [cdc](https://www.cdc.gov/growthcharts/cdc-data-files.htm)

## Example

If a child has \(X\) above \(M\), the z-score becomes positive, so the percentile rises above 50. If \(X\) is below \(M\), the z-score is negative, so the percentile falls below 50. [peteredewitt](https://www.peteredewitt.com/pedbp/articles/growth-standards.html)

For your VO2 table, the practical workflow is:
1. pick the row for the child’s age,
2. get \(L, M, S\),
3. compute \(z\),
4. map \(z\) to percentile with the normal CDF. [cdc](https://www.cdc.gov/growthcharts/cdc-data-files.htm)

## Useful reference points

Common z-to-percentile anchors are:
- \(z = 0\) → 50th percentile.
- \(z \approx 1.645\) → 95th percentile.
- \(z \approx -1.645\) → 5th percentile. [cdc](https://www.cdc.gov/growthcharts/cdc-data-files.htm)

If you want, I can next write the exact Python function for your VO2 LMS dataframe that returns both the z-score and percentile.

Percentiles from LMS are computed in two steps: first convert the observed value \(X\) to a z-score using the age-specific \(L, M, S\) values, then convert that z-score to a percentile using the standard normal distribution. [cdc](https://www.cdc.gov/growthcharts/cdc-data-files.htm)


In [ ]:
import pandas as pd
import numpy as np
import streamlit as st
from scipy.stats import norm

import pandas as pd

import pandas as pd

def load_vo2_girls_lms():
    data = [
        (8.0, 0.7265509, 39.58156, 0.1121695),
        (8.5, 0.7265509, 39.81146, 0.1145960),
        (9.0, 0.7265509, 40.04136, 0.1170225),
        (9.5, 0.7265509, 40.257865, 0.11944925),
        (10.0, 0.7265509, 40.47437, 0.1218760),
        (10.5, 0.7265509, 40.64493, 0.12430425),
        (11.0, 0.7265509, 40.81549, 0.1267325),
        (11.5, 0.7265509, 40.96958, 0.1291837),
        (12.0, 0.7265509, 41.12367, 0.1316349),
        (12.5, 0.7265509, 41.273085, 0.1341266),
        (13.0, 0.7265509, 41.4225, 0.1366183),
        (13.5, 0.7265509, 41.55895, 0.13915805),
        (14.0, 0.7265509, 41.6954, 0.1416978),
        (14.5, 0.7265509, 41.78817, 0.1442881),
        (15.0, 0.7265509, 41.88094, 0.1468784),
        (15.5, 0.7265509, 41.91419, 0.1495051),
        (16.0, 0.7265509, 41.94744, 0.1521318),
        (16.5, 0.7265509, 41.908465, 0.1547792),
        (17.0, 0.7265509, 41.86949, 0.1574266),
        (17.5, 0.7265509, 41.782325, 0.16008035),
        (18.0, 0.7265509, 41.69516, 0.1627341),
    ]

    df = pd.DataFrame(data, columns=["Age", "L", "M", "S"])
    return df

df_girls = load_vo2_girls_lms()
df_girls

def load_vo2_boys_lms():
    data = [
        (8.0, 0.7198903, 45.60843, 0.1225699),
        (8.5, 0.7198903, 46.10422, 0.1227899),
        (9.0, 0.7198903, 46.60001, 0.1230099),
        (9.5, 0.7198903, 47.054605, 0.12322995),
        (10.0, 0.7198903, 47.5092, 0.12345),
        (10.5, 0.7198903, 47.851825, 0.12367),
        (11.0, 0.7198903, 48.19445, 0.12389),
        (11.5, 0.7198903, 48.43505, 0.12411),
        (12.0, 0.7198903, 48.67565, 0.12433),
        (12.5, 0.7198903, 48.87173, 0.12455),
        (13.0, 0.7198903, 49.06781, 0.12477),
        (13.5, 0.7198903, 49.19721, 0.12499),
        (14.0, 0.7198903, 49.32661, 0.12521),
        (14.5, 0.7198903, 49.32027, 0.12543),
        (15.0, 0.7198903, 49.31393, 0.12565),
        (15.5, 0.7198903, 49.172395, 0.12587),
        (16.0, 0.7198903, 49.03086, 0.12609),
        (16.5, 0.7198903, 48.805915, 0.12631005),
        (17.0, 0.7198903, 48.58097, 0.1265301),
        (17.5, 0.7198903, 48.30935, 0.1267501),
        (18.0, 0.7198903, 48.03773, 0.1269701),
    ]

    df = pd.DataFrame(data, columns=["Age", "L", "M", "S"])
    df["Age"] = df["Age"].astype(float)
    df["L"] = df["L"].astype(float)
    df["M"] = df["M"].astype(float)
    df["S"] = df["S"].astype(float)
    return df

# Usage
df_vo2 = load_vo2_boys_lms()
# print(df_vo2)

def vo2_percentile_from_lms(age, observed_vo2, df):
    """
    df must contain columns: Age, L, M, S
    """
    row = df.loc[df["Age"] == age]

    if row.empty:
        raise ValueError(f"No LMS row found for age={age}")

    L = float(row["L"].iloc[0])
    M = float(row["M"].iloc[0])
    S = float(row["S"].iloc[0])

    if observed_vo2 <= 0:
        raise ValueError("observed_vo2 must be > 0")

    if L == 0:
        z = np.log(observed_vo2 / M) / S
    else:
        z = (((observed_vo2 / M) ** L) - 1) / (L * S)

    percentile = norm.cdf(z) * 100

    return {
        "age": age,
        "observed_vo2": observed_vo2,
        "L": L,
        "M": M,
        "S": S,
        "z_score": z,
        "percentile": percentile
    }

import numpy as np
import pandas as pd
from scipy.stats import norm

def vo2_percentile_from_lms_interp(age, observed_vo2, df):
    """
    df must contain columns: Age, L, M, S
    age can be any value within the table range
    """
    if observed_vo2 <= 0:
        raise ValueError("observed_vo2 must be > 0")

    df = df.sort_values("Age").reset_index(drop=True)

    ages = df["Age"].to_numpy(dtype=float)
    L_vals = df["L"].to_numpy(dtype=float)
    M_vals = df["M"].to_numpy(dtype=float)
    S_vals = df["S"].to_numpy(dtype=float)

    if age < ages.min() or age > ages.max():
        raise ValueError(f"Age must be between {ages.min()} and {ages.max()}")

    L = np.interp(age, ages, L_vals)
    M = np.interp(age, ages, M_vals)
    S = np.interp(age, ages, S_vals)

    if np.isclose(L, 0):
        z = np.log(observed_vo2 / M) / S
    else:
        z = (((observed_vo2 / M) ** L) - 1) / (L * S)

    percentile = norm.cdf(z) * 100

    return {
        "age": age,
        "observed_vo2": observed_vo2,
        "L": L,
        "M": M,
        "S": S,
        "z_score": z,
        "percentile": percentile
    }

def percentile_label(percentile):
    if percentile < 3:
        return "<P3"
    elif percentile > 97:
        return ">P97"
    else:
        return f"P{round(percentile)}"
    
df = pd.read_csv("vo2_boys_lms.csv")   # columns: Age, L, M, S

result = vo2_percentile_from_lms_interp(age=10.0, observed_vo2=52.0, df=df)

print(result["z_score"])
print(result["percentile"])

import numpy as np
from scipy.stats import norm

def get_vo2_percentile(age, observed_vo2, ref_df):
    ref_df = ref_df.sort_values("Age")

    L = np.interp(age, ref_df["Age"], ref_df["L"])
    M = np.interp(age, ref_df["Age"], ref_df["M"])
    S = np.interp(age, ref_df["Age"], ref_df["S"])

    if np.isclose(L, 0):
        z = np.log(observed_vo2 / M) / S
    else:
        z = (((observed_vo2 / M) ** L) - 1) / (L * S)

    percentile = norm.cdf(z) * 100
    return round(percentile, 2), round(z, 3)

percentile, z = get_vo2_percentile(age=12.3, observed_vo2=46.5, ref_df=df_vo2)
print(f"VO2 Percentile: {percentile:.2f}%, Z-score: {z:.2f}")

VO2 Percentile: 35.19%, Z-score: -0.38


In [16]:
import pandas as pd

df = pd.DataFrame({
    "Age": [8, 8.5, 9, 9.5, 10, 10.5, 11, 11.5, 12, 12.5, 13, 13.5, 14, 14.5, 15, 15.5, 16, 16.5, 17, 17.5, 18],
    "L": [-0.103452] * 21,
    "M": [2.93701, 2.9852215, 3.033433, 3.080116, 3.126799, 3.168747, 3.210695, 3.2499295, 3.289164, 3.32882, 3.368476, 3.406342, 3.444208, 3.476059, 3.50791, 3.5329195, 3.557929, 3.576667, 3.595405, 3.6108345, 3.626264],
    "S": [0.1662752, 0.16665645, 0.1670377, 0.1674114, 0.1677851, 0.168146, 0.1685069, 0.1688604, 0.1692139, 0.1695556, 0.1698973, 0.1702337, 0.1705701, 0.17092045, 0.1712708, 0.17162835, 0.1719859, 0.1723472, 0.1727085, 0.17306655, 0.1734246]
})
df.to_csv('reference_values/WR_peak_kg_girls_lms.csv', index=False)

import pandas as pd

df = pd.DataFrame({
    "Age": [8.16, 8.58, 9, 9.5, 10, 10.5, 11, 11.5, 12, 12.5, 13, 13.5, 14, 14.5, 15, 15.5, 16, 16.5, 17, 17.5, 18],
    "L": [-0.6918437] * 21,
    "M": [3.208293, 3.2814565, 3.35462, 3.435897, 3.517174, 3.588249, 3.659324, 3.7201335, 3.780943, 3.8319855, 3.883028, 3.919354, 3.95568, 3.9715275, 3.987375, 3.986894, 3.986413, 3.9792205, 3.972028, 3.9623055, 3.952583],
    "S": [0.1766309, 0.1751922, 0.1737535, 0.17204075, 0.170328, 0.1686153, 0.1669026, 0.16518985, 0.1634771, 0.16176435, 0.1600516, 0.15833885, 0.1566261, 0.1549134, 0.1532007, 0.15148795, 0.1497752, 0.14806245, 0.1463497, 0.14463695, 0.1429242]
})

df.to_csv('reference_values/WR_peak_kg_boys_lms.csv', index=False)


In [ ]:
def load_vo2_girls_lms():
    data = [
        (8.0, 0.7265509, 39.58156, 0.1121695),
        (8.5, 0.7265509, 39.81146, 0.1145960),
        (9.0, 0.7265509, 40.04136, 0.1170225),
        (9.5, 0.7265509, 40.257865, 0.11944925),
        (10.0, 0.7265509, 40.47437, 0.1218760),
        (10.5, 0.7265509, 40.64493, 0.12430425),
        (11.0, 0.7265509, 40.81549, 0.1267325),
        (11.5, 0.7265509, 40.96958, 0.1291837),
        (12.0, 0.7265509, 41.12367, 0.1316349),
        (12.5, 0.7265509, 41.273085, 0.1341266),
        (13.0, 0.7265509, 41.4225, 0.1366183),
        (13.5, 0.7265509, 41.55895, 0.13915805),
        (14.0, 0.7265509, 41.6954, 0.1416978),
        (14.5, 0.7265509, 41.78817, 0.1442881),
        (15.0, 0.7265509, 41.88094, 0.1468784),
        (15.5, 0.7265509, 41.91419, 0.1495051),
        (16.0, 0.7265509, 41.94744, 0.1521318),
        (16.5, 0.7265509, 41.908465, 0.1547792),
        (17.0, 0.7265509, 41.86949, 0.1574266),
        (17.5, 0.7265509, 41.782325, 0.16008035),
        (18.0, 0.7265509, 41.69516, 0.1627341),
    ]

    df = pd.DataFrame(data, columns=["Age", "L", "M", "S"])
    return df

df_girls = load_vo2_girls_lms()
df_girls.to_csv('reference_values/vo2peak_kg_girls_lms.csv', index=False)

def load_vo2_boys_lms():
    data = [
        (8.0, 0.7198903, 45.60843, 0.1225699),
        (8.5, 0.7198903, 46.10422, 0.1227899),
        (9.0, 0.7198903, 46.60001, 0.1230099),
        (9.5, 0.7198903, 47.054605, 0.12322995),
        (10.0, 0.7198903, 47.5092, 0.12345),
        (10.5, 0.7198903, 47.851825, 0.12367),
        (11.0, 0.7198903, 48.19445, 0.12389),
        (11.5, 0.7198903, 48.43505, 0.12411),
        (12.0, 0.7198903, 48.67565, 0.12433),
        (12.5, 0.7198903, 48.87173, 0.12455),
        (13.0, 0.7198903, 49.06781, 0.12477),
        (13.5, 0.7198903, 49.19721, 0.12499),
        (14.0, 0.7198903, 49.32661, 0.12521),
        (14.5, 0.7198903, 49.32027, 0.12543),
        (15.0, 0.7198903, 49.31393, 0.12565),
        (15.5, 0.7198903, 49.172395, 0.12587),
        (16.0, 0.7198903, 49.03086, 0.12609),
        (16.5, 0.7198903, 48.805915, 0.12631005),
        (17.0, 0.7198903, 48.58097, 0.1265301),
        (17.5, 0.7198903, 48.30935, 0.1267501),
        (18.0, 0.7198903, 48.03773, 0.1269701),
    ]

    df = pd.DataFrame(data, columns=["Age", "L", "M", "S"])
    df["Age"] = df["Age"].astype(float)
    df["L"] = df["L"].astype(float)
    df["M"] = df["M"].astype(float)
    df["S"] = df["S"].astype(float)
    return df

# Usage
df_vo2 = load_vo2_boys_lms()
df_vo2.to_csv('reference_values/vo2peak_kg_boys_lms.csv', index=False)

import pandas as pd

df_girls = pd.DataFrame({
    "Age": [6, 6.5, 7, 7.5, 8, 8.5, 9, 9.5, 10, 10.5, 11, 11.5, 12, 12.5, 13, 13.5, 14, 14.5, 15, 15.5, 16, 16.5, 17, 17.5, 18],
    "L":   [1.268, 1.262, 1.256, 1.251, 1.250, 1.254, 1.262, 1.273, 1.282, 1.283, 1.272, 1.244, 1.198, 1.133, 1.053, 0.963, 0.873, 0.793, 0.729, 0.680, 0.639, 0.603, 0.569, 0.534, 0.591],
    "M":   [0.366, 0.368, 0.369, 0.371, 0.373, 0.374, 0.375, 0.376, 0.377, 0.377, 0.378, 0.379, 0.380, 0.381, 0.382, 0.383, 0.384, 0.385, 0.385, 0.386, 0.387, 0.387, 0.388, 0.388, 0.391],
    "S":   [0.112, 0.111, 0.109, 0.108, 0.107, 0.106, 0.105, 0.105, 0.105, 0.105, 0.106, 0.107, 0.108, 0.110, 0.111, 0.113, 0.114, 0.115, 0.116, 0.117, 0.117, 0.118, 0.118, 0.118, 0.117]
})

df_girls.to_csv('reference_values/cimt_age_girls_LMS.csv', index = False)

import pandas as pd

df_height = pd.DataFrame({
    "Height (cm)": [120, 125, 130, 135, 140, 145, 150, 155, 160, 165, 170, 175, 180],
    "L": [1.585, 1.588, 1.586, 1.571, 1.533, 1.462, 1.338, 1.137, 0.872, 0.604, 0.378, 0.169, -0.043],
    "M": [0.369, 0.372, 0.375, 0.377, 0.378, 0.380, 0.380, 0.381, 0.383, 0.384, 0.385, 0.386, 0.387],
    "S": [0.108, 0.106, 0.105, 0.104, 0.104, 0.105, 0.107, 0.110, 0.113, 0.117, 0.121, 0.124, 0.127]
})


df_height.to_csv('reference_values/cimt_height_girls_LMS.csv', index = False)


import pandas as pd

# Age-specific boys cIMT LMS
df_cimt_boys_age = pd.DataFrame({
    "Age (years)": [6, 6.5, 7, 7.5, 8, 8.5, 9, 9.5, 10, 10.5, 11, 11.5, 12, 12.5, 13, 13.5, 14, 14.5, 15, 15.5, 16, 16.5, 17, 17.5, 18],
    "L": [1.444, 1.377, 1.309, 1.239, 1.165, 1.086, 1.004, 0.925, 0.851, 0.787, 0.735, 0.694, 0.667, 0.655, 0.664, 0.695, 0.746, 0.813, 0.890, 0.970, 1.048, 1.124, 1.198, 1.271, 1.342],
    "M": [0.370, 0.371, 0.372, 0.373, 0.374, 0.375, 0.376, 0.377, 0.378, 0.379, 0.380, 0.381, 0.382, 0.384, 0.386, 0.389, 0.391, 0.393, 0.396, 0.398, 0.400, 0.402, 0.404, 0.406, 0.408],
    "S": [0.121, 0.121, 0.121, 0.121, 0.120, 0.120, 0.119, 0.119, 0.118, 0.118, 0.118, 0.117, 0.117, 0.117, 0.118, 0.118, 0.118, 0.119, 0.119, 0.119, 0.119, 0.119, 0.119, 0.119, 0.120]
})

# Height-specific boys cIMT LMS
df_cimt_boys_height = pd.DataFrame({
    "Height (cm)": [120, 125, 130, 135, 140, 145, 150, 155, 160, 165, 170, 175, 180, 185, 190],
    "L": [1.274, 1.234, 1.188, 1.133, 1.076, 1.020, 0.966, 0.908, 0.840, 0.774, 0.714, 0.670, 0.650, 0.662, 0.694],
    "M": [0.369, 0.372, 0.375, 0.377, 0.379, 0.381, 0.384, 0.386, 0.389, 0.392, 0.395, 0.398, 0.401, 0.403, 0.405],
    "S": [0.119, 0.119, 0.119, 0.119, 0.120, 0.120, 0.121, 0.121, 0.121, 0.120, 0.119, 0.118, 0.117, 0.115, 0.114]
})

df_cimt_boys_age
df_cimt_boys_age.to_csv('reference_values/cimt_age_boys_LMS.csv', index = False)
df_cimt_boys_height
df_cimt_boys_height.to_csv('reference_values/cimt_height_boys_LMS.csv', index = False)



In [15]:
import numpy as np
import pandas as pd
from scipy.stats import norm


def percentile_label(percentile: float) -> str:
    if percentile < 3:
        return "<P3"
    if percentile > 97:
        return ">P97"
    return f"P{round(percentile)}"


def load_reference_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [col.strip() for col in df.columns]
    return df


def get_interp_lms(x: float, ref_df: pd.DataFrame, x_col: str) -> tuple[float, float, float]:
    df = ref_df.sort_values(x_col).reset_index(drop=True)

    x_vals = df[x_col].to_numpy(dtype=float)
    l_vals = df["L"].to_numpy(dtype=float)
    m_vals = df["M"].to_numpy(dtype=float)
    s_vals = df["S"].to_numpy(dtype=float)

    if x < x_vals.min() or x > x_vals.max():
        raise ValueError(f"{x_col} must be between {x_vals.min()} and {x_vals.max()}")

    l = np.interp(x, x_vals, l_vals)
    m = np.interp(x, x_vals, m_vals)
    s = np.interp(x, x_vals, s_vals)

    return float(l), float(m), float(s)


def lms_to_zscore(value: float, l: float, m: float, s: float) -> float:
    if value <= 0:
        raise ValueError("Observed value must be > 0")

    if np.isclose(l, 0):
        return float(np.log(value / m) / s)

    return float((((value / m) ** l) - 1) / (l * s))


def zscore_to_percentile(z: float) -> float:
    return float(norm.cdf(z) * 100)


def lms_score(value: float, x: float, ref_df: pd.DataFrame, x_col: str) -> dict:
    l, m, s = get_interp_lms(x=x, ref_df=ref_df, x_col=x_col)
    z = lms_to_zscore(value=value, l=l, m=m, s=s)
    percentile = zscore_to_percentile(z)

    return {
        "x_col": x_col,
        "x_value": float(x),
        "observed_value": float(value),
        "L": l,
        "M": m,
        "S": s,
        "z_score": z,
        "percentile": percentile,
        "percentile_label": percentile_label(percentile),
    }


def get_reference_df(refs: dict, metric: str, sex: str, reference_type: str) -> pd.DataFrame:
    try:
        path = refs[metric][sex][reference_type]
    except KeyError:
        raise ValueError(
            f"Missing reference for metric={metric}, sex={sex}, reference_type={reference_type}"
        )
    return load_reference_csv(path)


def score_vo2(sex: str, observed_value: float, age: float, refs: dict) -> dict:
    ref_df = get_reference_df(refs, metric="vo2", sex=sex, reference_type="age")
    result = lms_score(
        value=observed_value,
        x=age,
        ref_df=ref_df,
        x_col="Age",
    )
    result.update({
        "metric": "vo2",
        "sex": sex,
        "reference_type": "age",
    })
    return result


def score_cimt(sex: str, observed_value: float, refs: dict, age: float = None, height: float = None) -> dict:
    results = {
        "metric": "cimt",
        "sex": sex,
        "observed_value": float(observed_value),
    }

    if age is not None:
        age_df = get_reference_df(refs, metric="cimt", sex=sex, reference_type="age")
        results["age_based"] = lms_score(
            value=observed_value,
            x=age,
            ref_df=age_df,
            x_col="Age (years)",
        )

    if height is not None:
        height_df = get_reference_df(refs, metric="cimt", sex=sex, reference_type="height")
        results["height_based"] = lms_score(
            value=observed_value,
            x=height,
            ref_df=height_df,
            x_col="Height (cm)",
        )

    if "age_based" not in results and "height_based" not in results:
        raise ValueError("cIMT scoring requires age and/or height")

    return results


def score_measurement(
    metric: str,
    sex: str,
    observed_value: float,
    refs: dict,
    age: float = None,
    height: float = None,
) -> dict:
    metric = metric.strip().lower()
    sex = sex.strip().lower()

    if metric == "vo2":
        if age is None:
            raise ValueError("VO2 scoring requires age")
        return score_vo2(
            sex=sex,
            observed_value=observed_value,
            age=age,
            refs=refs,
        )

    if metric == "cimt":
        return score_cimt(
            sex=sex,
            observed_value=observed_value,
            refs=refs,
            age=age,
            height=height,
        )

    raise ValueError(f"Unsupported metric: {metric}")


refs = {
    "vo2": {
        "boys": {"age": "reference_values/vo2peak_kg_boys_lms.csv"},
        "girls": {"age": "reference_values/vo2peak_kg_girls_lms.csv"},
    },
    "cimt": {
        "boys": {
            "age": "reference_values/cimt_age_boys_LMS.csv",
            "height": "reference_values/cimt_height_boys_LMS.csv",
        },
        "girls": {
            "age": "reference_values/cimt_age_girls_LMS.csv",
            "height": "reference_values/cimt_height_girls_LMS.csv",
        },
    },
}

vo2_result = score_measurement(
    metric="vo2",
    sex="boys",
    observed_value=52.0,
    age=10.0,
    refs=refs,
)

print(vo2_result)

cimt_result = score_measurement(
    metric="cimt",
    sex="boys",
    observed_value=0.395,
    age=14.2,
    height=162,
    refs=refs,
)

print(cimt_result["age_based"])
print(cimt_result["height_based"])

{'x_col': 'Age', 'x_value': 10.0, 'observed_value': 52.0, 'L': 0.7198903, 'M': 47.5092, 'S': 0.12345, 'z_score': 0.7559447468941503, 'percentile': 77.51588380276208, 'percentile_label': 'P78', 'metric': 'vo2', 'sex': 'boys', 'reference_type': 'age'}
{'x_col': 'Age (years)', 'x_value': 14.2, 'observed_value': 0.395, 'L': 0.7727999999999999, 'M': 0.39180000000000004, 'S': 0.11839999999999999, 'z_score': 0.06891790251028429, 'percentile': 52.7472515871071, 'percentile_label': 'P53'}
{'x_col': 'Height (cm)', 'x_value': 162.0, 'observed_value': 0.395, 'L': 0.8136, 'M': 0.3902, 'S': 0.1206, 'z_score': 0.10188514640883208, 'percentile': 54.05760799736123, 'percentile_label': 'P54'}
